# W12 · Lab B · Train, Edit and Explore a Small VAE

Advanced Machine Learning for Artificial Intelligence · November 17, 2026

Use CPU. Save your own Colab copy before editing. This notebook generates data from two fixed seeds and needs no upload or credentials. Each model sees the same training images. Keep the held-out split unchanged. Follow the note's sections S05–S07.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "torch==2.8.0",
        "--index-url", "https://download.pytorch.org/whl/cpu"
    ])


In [ ]:
RELEASE = 'https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12'
EXPECTED_MODULES = {'advml_w12.py': '35c9e6b9083341d768976feb55c08ed5b4f57e4aa4108b452611dce04b213218', 'advml_w12_app.py': '9b3f983ab0a1b22d9dea7a9e7540e9a4593f19d15492038cbca01cb578d0b667'}


In [ ]:
import hashlib
from pathlib import Path
import sys
import tempfile
from urllib.request import urlopen

source_work = tempfile.TemporaryDirectory(prefix="advml-w12-")
source_cache = Path(source_work.name)
for name, expected in EXPECTED_MODULES.items():
    local_paths = [Path.cwd() / "src" / name]
    data = next((p.read_bytes().replace(b"\r\n", b"\n") for p in local_paths if p.is_file()), None)
    if data is None:
        with urlopen(f"{RELEASE}/src/{name}", timeout=30) as response:
            data = response.read().replace(b"\r\n", b"\n")
    assert hashlib.sha256(data).hexdigest() == expected, f"Source hash mismatch: {name}"
    (source_cache / name).write_bytes(data)
sys.path.insert(0, str(source_cache.resolve()))


## S05 — The VAE Objective and the AE Baseline

**B01 · Locate data, parameters, and logits.** Inspect one dataset row and the learned model's output shape. Predict which axis corresponds to the 256 Bernoulli observations.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F06-observations.png" alt="Locate data, parameters, and logits" width="820">


In [ ]:
import csv
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from advml_w12 import build_model, decoded_means, evaluate_model, mean_codes, shape_images, train_model
from advml_w12_app import decoder_view, export_view

train, _ = shape_images(1024, 248)
heldout, heldout_factors = shape_images(192, 905)
x0 = torch.tensor(train[:2])
probe = build_model()
mu0, lv0 = probe.encode(x0)
print("input, encoder mean, log variance, decoder logits")
print(x0.shape, mu0.shape, lv0.shape, probe.decoder(mu0).shape)
plt.imshow(heldout[0].reshape(16,16), vmin=0, vmax=1, cmap="Blues")
plt.title("One held-out binary observation")
plt.axis("off")
plt.show()


## S05 — The VAE Objective and the AE Baseline

**B02 · Read and edit the actual training loss.** Trace the pixel sum, latent sum, and batch mean. To see a failing check, temporarily change the pixel sum to mean in vae_objective and rerun the check. Restore it before training.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F05-objective.png" alt="Read and edit the actual training loss" width="820">


In [ ]:
def reparameterize(mu, logvar, epsilon):
    """Same-shaped tensors with z = mu + sigma*epsilon and a fixed base draw."""
    if mu.shape != logvar.shape or mu.shape != epsilon.shape:
        raise ValueError("mu, logvar, and epsilon must share [batch, latent] shape.")
    sigma = (0.5 * logvar).exp()
    return mu + sigma * epsilon


def diagonal_kl(mu, logvar):
    """One KL per image, summing latent coordinates."""
    if mu.ndim != 2 or mu.shape != logvar.shape:
        raise ValueError("Expected [batch, latent] arrays with identical shape.")
    return 0.5 * (mu.square() + logvar.exp() - 1.0 - logvar).sum(dim=-1)


def vae_objective(logits, x, mu, logvar, beta=1.0, *, kl_fn=diagonal_kl):
    """Bernoulli NLL sums pixels, KL sums latents, both retain batch until the end."""
    import torch.nn.functional as F

    if logits.ndim != 2 or logits.shape != x.shape or len(x) != len(mu):
        raise ValueError("Expected [batch, pixels] logits and aligned latent rows.")
    if not math.isfinite(beta) or beta < 0:
        raise ValueError("beta must be finite and nonnegative.")
    nll = F.binary_cross_entropy_with_logits(logits, x, reduction="none").sum(dim=-1)
    kl = kl_fn(mu, logvar)
    if kl.shape != nll.shape:
        raise ValueError("NLL and KL must each have shape [batch].")
    return (nll + beta * kl).mean(), nll, kl


In [ ]:
assert train.shape == (1024, 256) and heldout.shape == (192, 256)
assert np.isin(train, [0., 1.]).all() and not np.array_equal(train[:len(heldout)], heldout)
mu = torch.tensor([[1.]], dtype=torch.float64, requires_grad=True)
lv = torch.tensor([[math.log(.25)]], dtype=torch.float64, requires_grad=True)
z = reparameterize(mu, lv, torch.tensor([[-2.]], dtype=torch.float64))
dmu, dlv = torch.autograd.grad(z.sum(), [mu, lv])
np.testing.assert_allclose([z.item(),dmu.item(),dlv.item()], [0.,1.,-.5], atol=1e-10)
logits = torch.zeros([1,2], dtype=torch.float64)
target = torch.tensor([[0.,1.]], dtype=torch.float64)
loss, nll, kl = vae_objective(logits, target, mu, lv, 1.)
np.testing.assert_allclose([nll.item(),kl.item(),loss.item()],
    [2*math.log(2), .81814718056, 2*math.log(2)+.81814718056], atol=1e-10)
print(f"Two-pixel check: NLL={nll.item():.6f}, KL={kl.item():.6f}, total={loss.item():.6f}")


## S06 — Train, Change One Line, and Compare

**B03 · Fit on the same split and compare the same rows.** These three fits receive the actual draw and loss functions from B02. The output strips use posterior means for the VAE. The reported NLL estimates the posterior expectation over four draws.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F07-reconstruction.png" alt="Fit on the same split and compare the same rows" width="820">


In [ ]:
EPOCHS = 15
FIT_SEED = 1742
controls = [("AE", 0., False), ("VAE beta=.25", .25, True), ("VAE beta=1", 1., True)]

def fit(beta, variational=True):
    model, history = train_model(train, beta=beta, variational=variational, epochs=EPOCHS,
        seed=FIT_SEED, draw_fn=reparameterize, objective_fn=vae_objective)
    return model, evaluate_model(model, heldout), history

fits = {label: fit(beta, variational) for label, beta, variational in controls}
for label, (_, metrics, _) in fits.items():
    print(label, metrics)

rows = [0,1,2,3,5,6,7,8]
images = [heldout[rows].reshape(-1,16,16)]
images += [decoded_means(m,mean_codes(m,heldout[rows])) for m,_,_ in fits.values()]
fig, axes = plt.subplots(len(images),len(rows),figsize=(10,5),layout="constrained")
for r, (label, group) in enumerate(zip(["Observed", *fits], images)):
    for c, im in enumerate(group):
        axes[r,c].imshow(im,vmin=0,vmax=1,cmap="Blues")
        axes[r,c].axis("off")
        if c==0:
            axes[r,c].set_title(label,fontsize=9)
plt.show()


## S06 — Train, Change One Line, and Compare

**B04 · Predict, change one line, and compare.** Predict whether the held-out NLL and KL will increase or decrease at beta=4. Edit TRIAL_BETA, keep the other settings fixed, and record both columns with units.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F08-metrics.png" alt="Predict, change one line, and compare" width="820">


In [ ]:
TRIAL_BETA = .25  # Change this line to 4.0 after committing to a prediction.
trial_model, trial_metrics, _ = fit(TRIAL_BETA)
print("beta=1      ", fits["VAE beta=1"][1])
print(f"beta={TRIAL_BETA:<5} ", trial_metrics)


## S07 — Explore the Latent Space and Check the Claim

**B05 · Choose z three ways.** Hold the decoder fixed. Identify which row requires an input at generation time. Edit the pair of endpoint indices, then distinguish interpolation from sampling.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F09-decode.png" alt="Choose z three ways" width="820">


In [ ]:
model, metrics, history = fits["VAE beta=1"]
ENDPOINTS = [0,8]
endpoints = mean_codes(model, heldout[ENDPOINTS])
codes = [mean_codes(model,heldout[:8]), np.random.default_rng(308).normal(size=(8,2)),
         np.linspace(*endpoints,8)]
fig, axes = plt.subplots(3,8,figsize=(11,4),layout="constrained")
for row, (label, zs) in enumerate(zip(["Posterior mean","Prior draws","Interpolation"], codes)):
    for col, im in enumerate(decoded_means(model,zs)):
        axes[row,col].imshow(im,vmin=0,vmax=1,cmap="Blues")
        axes[row,col].axis("off")
        if col==0:
            axes[row,col].set_title(label,fontsize=9)
plt.show()


## S07 — Explore the Latent Space and Check the Claim

**B06 · Run the decoder explorer.** Move one coordinate while holding the other fixed. Compare the probability map and a binary observation. Export a CSV row for each coordinate you tested. No external app tunnel is needed.


<img src="https://raw.githubusercontent.com/lunalab-ai/advML/2026-fall-w12/course/notion/w12/assets/F10-latent.png" alt="Run the decoder explorer" width="820">


In [ ]:
from IPython.display import FileLink, clear_output, display

def inspect_coordinate(z1, z2):
    # Edit this line to choose a different observation seed.
    return decoder_view(model, metrics, z1, z2, seed=941)

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    # Equivalent controls in a minimal local kernel.
    Z1, Z2 = -1., .5
    figure, row = inspect_coordinate(Z1,Z2)
    display(figure)
    plt.close(figure)
    print(row)
else:
    z1 = widgets.FloatSlider(value=0, min=-3, max=3, step=.1, description="z1", continuous_update=False)
    z2 = widgets.FloatSlider(value=0, min=-3, max=3, step=.1, description="z2", continuous_update=False)
    out = widgets.Output()
    button = widgets.Button(description="Export current row")

    def show(_=None):
        with out:
            clear_output(wait=True)
            figure, row = inspect_coordinate(z1.value, z2.value)
            display(figure)
            plt.close(figure)
            print(row)

    def export(_):
        with out:
            import pandas as pd

            _, row = inspect_coordinate(z1.value, z2.value)
            plt.close(plt.gcf())
            path = Path("w12-decoder-row.csv")
            pd.DataFrame([row]).to_csv(path, index=False)
            try:
                from google.colab import files
            except ImportError:
                display(FileLink(str(path)))
            else:
                files.download(str(path))

    z1.observe(show, names="value")
    z2.observe(show, names="value")
    button.on_click(export)
    display(widgets.VBox([widgets.HBox([z1,z2]),button,out]))
    show()


In [ ]:
first, origin = inspect_coordinate(0.,0.)
second, moved = inspect_coordinate(2.,0.)
plt.close(first)
plt.close(second)
assert origin["z1"] == 0 and moved["z1"] == 2
assert not np.allclose(decoded_means(model, [[0.,0.]]), decoded_means(model, [[2.,0.]]))
exported = Path(export_view(model,metrics,2.,0.))
with exported.open(newline="",encoding="utf-8") as stream:
    exported_row = next(csv.DictReader(stream))
np.testing.assert_allclose(float(exported_row["mean_pixel_probability"]),moved["mean_pixel_probability"])
exported.unlink()
assert all(np.isfinite(item[1]["heldout_nll_nats_per_pixel"]) for item in fits.values())
assert fits["AE"][1]["heldout_kl_nats_per_image"] is None
print("Checks passed: callback, coordinate sensitivity, CSV, held-out metrics.")
